### Basic working of Google Palm LLM in LangChain

In [ ]:
from langchain.chains import RetrievalQA
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.embeddings import HuggingFaceInstructEmbeddings
# Or if using Google Gemini Embeddings:
# from langchain_google_genai import GoogleGenerativeAIEmbeddings

### Now let's load data from Codebasics FAQ csv file

In [ ]:
# Updated import for modern LangChain
from langchain_community.document_loaders import CSVLoader

# Load the dataset using the exercise/food name as the source identifier
loader = CSVLoader(file_path='fitness_data.csv', source_column="name")

# Store the loaded document objects in the 'data' variable
data = loader.load()

# Optional: Inspect the first loaded document entry
print(data[0].page_content)
print("Source Metadata:", data[0].metadata)

### Hugging Face Embeddings

In [ ]:
len(e)

In [ ]:
e[:5]

As you can see above, embedding for a sentance "What is your refund policy" is a list of size 768. Looking at the numbers in this list, doesn't give any intuitive understanding of what it is but just assume that these numbers are capturing the meaning of "What is your refund policy". If you are curious to know about embeddings, go to youtube and search "codebasics word embeddings" and you will find bunch of videos with simple, intuitive explanations

### Vector store using FAISS

In [ ]:
# Modern import from langchain_community
from langchain_community.vectorstores import FAISS

# Create FAISS vector store from loaded documents and embeddings
vectordb = FAISS.from_documents(
    documents=data,
    embedding=instructor_embeddings
)

# Optional: Save locally to test the persistence logic matching langchain_helper.py
vectordb.save_local("faiss_fitness_index")

# Create a retriever with a similarity threshold
retriever = vectordb.as_retriever(score_threshold=0.7)

In [ ]:
# Modern invoke method replacing deprecated get_relevant_documents()
rdocs = retriever.invoke("What is the schedule for a 6-day Push Pull Legs split?")

# Display the returned document chunks
rdocs

As you can see above, the retriever that was created using FAISS and hugging face embedding is now capable of pulling relavant documents from our original CSV file knowledge store. This is very powerful and it will help us further in our project

##### Embeddings can be created using GooglePalm too. Also for vector database you can use chromadb as well as shown below. During our experimentation, we found hugging face embeddings and FAISS to be more appropriate for our use case

In [ ]:
# google_palm_embeddings = GooglePalmEmbeddings(google_api_key=api_key)

# from langchain.vectorstores import Chroma
# vectordb = Chroma.from_documents(data,
#                            embedding=google_palm_embeddings,
#                            persist_directory='./chromadb')
# vectordb.persist()

### Create RetrievalQA chain along with prompt template 🚀

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain.chains import RetrievalQA

# Custom prompt template matching your 4-column fitness schema
prompt_template = """Given the following context and a question, generate an answer based on this context only.
In the answer try to provide as much text as possible from the "details" and "information" sections in the source document context without making significant changes.
If the answer is not found in the context, kindly state "I don't know." Don't try to make up an answer.

CONTEXT: {context}

QUESTION: {question}"""

PROMPT = PromptTemplate(
    template=prompt_template, 
    input_variables=["context", "question"]
)

chain_type_kwargs = {"prompt": PROMPT}

# Initialize RetrievalQA chain matching production configuration
chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    input_key="query",
    return_source_documents=True,
    chain_type_kwargs=chain_type_kwargs
)

### We are all set 👍🏼 Let's ask some questions now

In [ ]:
# Query your QA chain
response = chain.invoke({"query": "What are the macros for Chicken Breast?"})

# Correct print syntax
print("Answer:", response["result"])
print("\nSource Documents:", response["source_documents"])

**As you can see above, the answer of question comes from two different FAQs within our csv file and it is able to pull those questions and merge them nicely**

In [ ]:
chain("Do you guys provide internship and also do you offer EMI payments?")

In [ ]:
chain("do you have javascript course?")

In [ ]:
chain("Do you have plans to launch blockchain course in future?")

In [ ]:
chain("should I learn power bi or tableau?")

In [ ]:
chain("I've a MAC computer. Can I use powerbi on it?")

In [ ]:
chain("I don't see power pivot. how can I enable it?")

In [ ]:
chain("What is the price of your machine learning course?")